# CNNs, Sequences, and Encoder-Decoders — Hands-On Notebook

Companion to: [CNNs](../docs/00_prerequisites/05_convolutional_neural_networks.md), [RNNs](../docs/00_prerequisites/06_sequence_modeling_and_rnns.md), [Encoder-Decoder](../docs/00_prerequisites/07_encoder_decoder_paradigm.md)

**What you'll build:**
- A 1D convolution from scratch
- A small CNN for image classification
- A vanilla RNN cell, then an LSTM
- A simple encoder-decoder for sequence reversal

## Section 1: Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Section 2: Convolution from Scratch

A convolution slides a small filter over the input, computing dot products at each position — like scanning with a magnifying glass.

In [ ]:
signal = np.array([1, 2, 3, 4, 5, 6], dtype=np.float32)
kernel = np.array([1, 0, -1], dtype=np.float32)
k = len(kernel)
out_len = len(signal) - k + 1
out_manual = np.zeros(out_len, dtype=np.float32)

print("1D convolution by hand (valid, stride 1)")
print("signal:", signal)
print("kernel:", kernel)
print()
for i in range(out_len):
    window = signal[i : i + k]
    dot = float(np.dot(window, kernel))
    out_manual[i] = dot
    print(f"t={i}: window={window} · kernel={kernel} = {dot:.4f}")

print("\nOutput (manual):", out_manual)

In [ ]:
x = torch.tensor(signal, dtype=torch.float32).view(1, 1, -1)
w = torch.tensor(kernel, dtype=torch.float32).view(1, 1, -1)
out_torch = F.conv1d(x, w, padding=0, stride=1)
print("Output (PyTorch):", out_torch.squeeze().detach().numpy())
print("Match:", np.allclose(out_manual, out_torch.squeeze().numpy()))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
t_sig = np.arange(len(signal))
axes[0].stem(t_sig, signal, basefmt=" ", linefmt="C0-", markerfmt="C0o")
axes[0].set_title("Input signal")
axes[0].set_ylabel("value")
axes[0].legend(["signal"], loc="upper right")

tk = np.arange(k)
axes[1].stem(tk, kernel, basefmt=" ", linefmt="C1-", markerfmt="C1s")
axes[1].set_title("Kernel")
axes[1].set_ylabel("weight")
axes[1].legend(["kernel"], loc="upper right")

t_out = np.arange(out_len)
axes[2].stem(t_out, out_manual, basefmt=" ", linefmt="C2-", markerfmt="C2^")
axes[2].set_title("Convolution output (length = N − K + 1)")
axes[2].set_xlabel("position")
axes[2].set_ylabel("value")
axes[2].legend(["output"], loc="upper right")
plt.tight_layout()
plt.show()

## Section 3: 2D Convolution & Pooling

We use a simple edge-like kernel on a checkerboard, then explore stride and padding.

In [ ]:
def make_checkerboard(h=64, w=64, n=8):
    """n×n grid of black/white squares."""
    ch = h // n
    cw = w // n
    img = np.zeros((h, w), dtype=np.float32)
    for i in range(n):
        for j in range(n):
            if (i + j) % 2 == 0:
                img[i * ch : (i + 1) * ch, j * cw : (j + 1) * cw] = 1.0
    return img

img_np = make_checkerboard()
edge_kernel = torch.tensor(
    [[[1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, -1.0]]],
    dtype=torch.float32,
).unsqueeze(0)

x2 = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0)
edges = F.conv2d(x2, edge_kernel, padding=0, stride=1)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im0 = ax[0].imshow(img_np, cmap="gray")
ax[0].set_title("Input (checkerboard)")
plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(edges.squeeze().numpy(), cmap="magma")
ax[1].set_title("Edge response (kernel [[1,0,-1],...])")
plt.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout()
plt.show()
print("Input shape:", tuple(x2.shape), "→ Edge map shape:", tuple(edges.shape))

In [ ]:
s2 = F.conv2d(x2, edge_kernel, padding=1, stride=2)
print("With padding=1, stride=2 → spatial size:", s2.shape[-2:])
print("(Smaller than valid stride=1 with padding=0 due to stride downsampling.)")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(edges.squeeze().numpy(), cmap="magma")
ax[0].set_title("stride=1, padding=0")
ax[1].imshow(s2.squeeze().numpy(), cmap="magma")
ax[1].set_title("stride=2, padding=1")
plt.tight_layout()
plt.show()

Pooling summarizes a region — like reading a paragraph and writing a one-line summary.

In [ ]:
pool_in = edges
mx = F.max_pool2d(pool_in, kernel_size=2, stride=2)
av = F.avg_pool2d(pool_in, kernel_size=2, stride=2)

print("Before pooling:", tuple(pool_in.shape))
print("After max_pool2d(2,2):", tuple(mx.shape))
print("After avg_pool2d(2,2):", tuple(av.shape))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, t, title in zip(ax, [pool_in, mx, av], ["input", "max pool", "avg pool"]):
    a.imshow(t.squeeze().numpy(), cmap="magma")
    a.set_title(title)
    a.axis("off")
plt.suptitle("Pooling reduces spatial size (here H,W halved)")
plt.tight_layout()
plt.show()

## Section 4: Small CNN on CIFAR-10

We train on the first 5k training images for speed. Notice how early filters learn edges and textures.

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)
full_train = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
subset_indices = list(range(5000))
train_set = torch.utils.data.Subset(full_train, subset_indices)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


model = SmallCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print(model)

In [ ]:
n_epochs = 5
for epoch in range(1, n_epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        opt.step()
        total_loss += loss.item() * imgs.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / total
    acc = correct / total
    print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | train_acc={acc:.4f}")

In [ ]:
w = model.conv1.weight.detach().cpu()
n_show = min(16, w.size(0))
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i in range(n_show):
    r, c = i // 4, i % 4
    filt = w[i].numpy().transpose(1, 2, 0)
    filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
    axes[r, c].imshow(filt)
    axes[r, c].axis("off")
    axes[r, c].set_title(f"filter {i}", fontsize=8)
plt.suptitle("First conv layer filters (RGB normalized per filter)")
plt.tight_layout()
plt.show()

## Section 5: Why Order Matters — The Sequence Problem

An MLP sees all inputs at once — it can't tell *dog bites man* from *man bites dog* without modeling order.

Below, the **same numbers** in **different orders** pass through a fixed `Linear` layer: outputs differ because each position has its own weights (order matters).

In [ ]:
torch.manual_seed(0)
mlp = nn.Linear(3, 1, bias=True)
x_order1 = torch.tensor([[1.0, 2.0, 3.0]])
x_order2 = torch.tensor([[3.0, 2.0, 1.0]])
y1 = mlp(x_order1)
y2 = mlp(x_order2)
print("Permutation 1 [1,2,3] →", y1.item())
print("Permutation 2 [3,2,1] →", y2.item())
print("Same multiset, different order → different output (not permutation invariant).")
print("Sum of features is permutation invariant:", x_order1.sum().item(), "vs", x_order2.sum().item())

## Section 6: Vanilla RNN from Scratch

RNNs process one step at a time, maintaining a hidden state — like reading a book word by word.

Update: \(h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b)\).

In [ ]:
input_dim, hidden_dim = 4, 5
T = 5
torch.manual_seed(42)
W_hh = torch.randn(hidden_dim, hidden_dim) * 0.3
W_xh = torch.randn(hidden_dim, input_dim) * 0.3
b = torch.zeros(hidden_dim)
h = torch.zeros(hidden_dim)
xs = [torch.randn(input_dim) for _ in range(T)]

print("Manual RNN forward")
hs_manual = []
for t, x in enumerate(xs):
    h = torch.tanh(W_hh @ h + W_xh @ x + b)
    hs_manual.append(h.clone())
    print(f"t={t}: ||h||={h.norm().item():.4f}")

rnn = nn.RNN(input_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
with torch.no_grad():
    rnn.weight_hh_l0.copy_(W_hh)
    rnn.weight_ih_l0.copy_(W_xh)
    rnn.bias_hh_l0.zero_()
    rnn.bias_ih_l0.copy_(b)
x_batch = torch.stack(xs).unsqueeze(0)
out, hn = rnn(x_batch)
print("\nMax diff vs torch RNN last hidden:", (hs_manual[-1] - hn.squeeze(0)).abs().max().item())

### The Vanishing Gradient Problem

Repeated multiplication by small gains shrinks signals (and gradients) across long chains.

In [ ]:
t_steps = np.arange(1, 101)
scale = 0.9 ** t_steps
plt.figure(figsize=(8, 4))
plt.semilogy(t_steps, scale, label=r"$0.9^t$")
plt.xlabel("t (steps)")
plt.ylabel("magnitude (log scale)")
plt.title("Vanishing signal: repeated scaling by 0.9")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"After 100 steps, 0.9^100 ≈ {0.9**100:.2e} — practically zero for many uses.")

## Section 7: LSTM — Gating to the Rescue

LSTM adds gates (forget, input, output) that control information flow — like a notebook where you can erase, write, and read selectively.

In [ ]:
def lstm_step(x_t, h_prev, c_prev, W_i, W_f, W_g, W_o, b_i, b_f, b_g, b_o):
    """One LSTM step with concatenated [h; x] for each gate (demo-sized)."""
    hx = torch.cat([h_prev, x_t], dim=0)
    i_t = torch.sigmoid(W_i @ hx + b_i)
    f_t = torch.sigmoid(W_f @ hx + b_f)
    g_t = torch.tanh(W_g @ hx + b_g)
    c_t = f_t * c_prev + i_t * g_t
    o_t = torch.sigmoid(W_o @ hx + b_o)
    h_t = o_t * torch.tanh(c_t)
    return h_t, c_t, i_t, f_t, o_t, g_t


idim, hdim = 4, 5
torch.manual_seed(123)
W_i = torch.randn(hdim, hdim + idim) * 0.25
W_f = torch.randn(hdim, hdim + idim) * 0.25
W_g = torch.randn(hdim, hdim + idim) * 0.25
W_o = torch.randn(hdim, hdim + idim) * 0.25
b_i = torch.zeros(hdim)
b_f = torch.zeros(hdim)
b_g = torch.zeros(hdim)
b_o = torch.zeros(hdim)

h = torch.zeros(hdim)
c = torch.zeros(hdim)
seq = [torch.randn(idim) for _ in range(5)]
print("Gate magnitudes (mean abs) per step:")
for t, x in enumerate(seq):
    h, c, i_t, f_t, o_t, g_t = lstm_step(x, h, c, W_i, W_f, W_g, W_o, b_i, b_f, b_g, b_o)
    print(
        f"t={t}: |i|={i_t.abs().mean():.3f} |f|={f_t.abs().mean():.3f} "
        f"|g|={g_t.abs().mean():.3f} |o|={o_t.abs().mean():.3f} ||h||={h.norm():.3f}"
    )

xb = torch.stack(seq).unsqueeze(0)
lstm_pt = nn.LSTM(idim, hdim, batch_first=True)
with torch.no_grad():
    out_pt, (hn_pt, cn_pt) = lstm_pt(xb)
print("\nVerify nn.LSTM: output shape", tuple(out_pt.shape), "| last h shape", tuple(hn_pt.shape))
print("(Manual gates above use explicit W matrices; nn.LSTM bundles the same math in weight_ih / weight_hh.)")

## Section 8: Character-Level LSTM on Tiny Text

Even this tiny model captures basic patterns like word boundaries and common letter combinations.

In [ ]:
text = (
    "The quick brown fox jumps over the lazy dog. " * 30
)[:1000]
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
V = len(chars)


class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        y, _ = self.lstm(e)
        return self.fc(y)


data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
seq_len = 32
X, Y = [], []
for i in range(0, len(data) - seq_len):
    X.append(data[i : i + seq_len])
    Y.append(data[i + 1 : i + seq_len + 1])
X = torch.stack(X)
Y = torch.stack(Y)
ds = torch.utils.data.TensorDataset(X, Y)
dl = DataLoader(ds, batch_size=64, shuffle=True)

cmodel = CharLSTM(V).to(device)
opt_c = torch.optim.Adam(cmodel.parameters(), lr=1e-2)
crit = nn.CrossEntropyLoss()
for ep in range(1, 6):
    cmodel.train()
    tot = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt_c.zero_grad()
        logits = cmodel(xb)
        loss = crit(logits.reshape(-1, V), yb.reshape(-1))
        loss.backward()
        opt_c.step()
        tot += loss.item()
    print(f"Char LSTM epoch {ep} | loss={tot/len(dl):.4f}")

cmodel.eval()
start = "The q"
idx = [stoi[c] for c in start]
h = None
gen = start
inp = torch.tensor([idx], dtype=torch.long, device=device)
with torch.no_grad():
    for _ in range(120):
        logits = cmodel(inp)
        last = logits[:, -1, :]
        probs = F.softmax(last, dim=-1)
        ni = torch.multinomial(probs, 1).item()
        gen += itos[ni]
        inp = torch.tensor([[ni]], dtype=torch.long, device=device)
print("\nSample generation:\n", gen)

## Section 9: Encoder-Decoder for Sequence Reversal

The encoder reads the input and compresses it to a single vector. The decoder reads that vector and generates the output — like taking notes during a lecture, then writing an essay from your notes.

This is the exact architecture that powered early machine translation (before attention).

In [ ]:
DIGITS = 10
SOS = 10
EOS = 11
VOCAB = DIGITS + 2
EMB = 32
HID = 64


class ReverseEncDec(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, EMB)
        self.enc = nn.LSTM(EMB, HID, batch_first=True)
        self.dec = nn.LSTM(EMB, HID, batch_first=True)
        self.fc = nn.Linear(HID, DIGITS)

    def forward(self, src, tgt_in):
        _, (h, c) = self.enc(self.emb(src))
        out, _ = self.dec(self.emb(tgt_in), (h, c))
        return self.fc(out)


def make_batch(batch_size=64, min_len=4, max_len=4):
    lengths = torch.randint(min_len, max_len + 1, (batch_size,))
    maxL = int(lengths.max())
    src = torch.zeros(batch_size, maxL, dtype=torch.long)
    for b in range(batch_size):
        L = int(lengths[b])
        src[b, :L] = torch.randint(0, DIGITS, (L,))
    rev = torch.zeros_like(src)
    for b in range(batch_size):
        L = int(lengths[b])
        rev[b, :L] = torch.flip(src[b, :L], dims=(0,))
    return src, rev, lengths


ed_model = ReverseEncDec().to(device)
ed_opt = torch.optim.Adam(ed_model.parameters(), lr=1e-3)
ed_loss_fn = nn.CrossEntropyLoss(reduction="none")

for epoch in range(1, 51):
    ed_model.train()
    losses = []
    for _ in range(40):
        src, rev, lengths = make_batch(64, 4, 4)
        src, rev = src.to(device), rev.to(device)
        B, maxL = src.shape
        sos = torch.full((B, 1), SOS, dtype=torch.long, device=device)
        tgt_in = torch.cat([sos, rev[:, :-1]], dim=1)
        logits = ed_model(src, tgt_in)
        mask = torch.arange(maxL, device=device).unsqueeze(0) < lengths.to(device).unsqueeze(1)
        loss_vec = ed_loss_fn(logits.reshape(-1, DIGITS), rev.reshape(-1))
        loss = (loss_vec.view(B, maxL) * mask.float()).sum() / mask.float().sum().clamp(min=1)
        ed_opt.zero_grad()
        loss.backward()
        ed_opt.step()
        losses.append(loss.item())
    if epoch % 10 == 0 or epoch == 1:
        print(f"Enc-Dec epoch {epoch:02d} | loss={np.mean(losses):.4f}")

In [ ]:
ed_model.eval()
tests = [
    [1, 2, 3, 4],
    [7, 0, 2, 9],
    [3, 3, 1, 5],
]
with torch.no_grad():
    for seq in tests:
        L = len(seq)
        src = torch.tensor([seq], dtype=torch.long, device=device)
        _, (h, c) = ed_model.enc(ed_model.emb(src))
        pred = []
        inp = torch.tensor([[SOS]], dtype=torch.long, device=device)
        hidden = (h, c)
        for _ in range(L):
            emb = ed_model.emb(inp)
            out, hidden = ed_model.dec(emb, hidden)
            logits = ed_model.fc(out[:, -1, :])
            next_tok = logits.argmax(dim=-1)
            pred.append(int(next_tok.item()))
            inp = next_tok.view(1, 1)
        exp = list(reversed(seq))
        print(f"input={seq} | predicted={pred} | expected={exp}")

## Section 10: Key Takeaways

- **Convolutions** detect local patterns; **pooling** builds invariance and reduces size.
- **MLPs** on flattened sequences mix positions only through fixed weights — order matters; **RNNs** carry state across time.
- **Vanishing gradients** make long-range credit assignment hard in deep RNNs; **LSTMs** use gating to keep paths for information and gradients.
- **Encoder–decoder** models map variable-length inputs to outputs via a compressed bottleneck; **attention** (in later topics) removes the strict single-vector bottleneck.

**Next:** experiment with longer reversal sequences, smaller models, or add attention — and compare training curves on CPU vs GPU if available.